# B2.10 · Severity calibration and reporting

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.9 · Remediation engineering](https://spbreed.github.io/cyber-commons/lessons/B2.9.html)**.

| | |
|---|---|
| Tools used | OpenGrep, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Recalculate severity from confirmed exploitation and reachability, then produce the per-stage escape economics.

**Why a security engineer needs it.** Severity is a label copied from the rule, so the queue is ordered by something that predicts nothing. The control it builds is: stage 15: calibrate severity from sandbox evidence, then report per-stage economics rather than a finding count.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

CVSS scores the vulnerability. Your engineers are asking about this system, with this data, behind this control, and the number that answers them is not the one on the badge.

> **At CyberTravels.** CVSS scores the vulnerability. CyberTravels' engineers are asking about this booking API, with card data, behind this gateway — and the number that answers them is not on the badge.

## 2 · The framework

```
   CVSS 9.8                    your system
   +----------------+          +---------------------------+
   | network        |          | internal only             |
   | no auth        |    vs    | behind SSO                |
   | full impact    |          | read-only replica         |
   +----------------+          +---------------------------+
          |                                |
      the badge                    the number engineers act on

   confirmed-by-exploitation beats both
```

**Stage 15 — Severity calibration and reporting.** The pipeline's output, and
the stage where its credibility is won or lost.

Most severity is a label copied from the rule that fired: this is a CWE-89, so
it is high. That number predicts nothing, because it ignores everything the
pipeline has just learned:

- did stage 12 **confirm it by execution**?
- is it **reachable** from an untrusted entry point (stage 10)?
- what does it **chain into** (stage 13)?
- does it sit in a **historical risk zone** (stage 1)?

Calibrated severity uses all four. A confirmed, reachable finding that chains
into account takeover is not the same as an unvalidated finding in dead code,
even when both are CWE-89.

The second half of this stage is the report, and the useful report is not a
finding count. It is **per-stage economics**: where bugs are caught, where they
escape, and what each escape costs — because that is what decides next
quarter's budget.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 15 as a skill — severity you can argue with

The reporting skill carries one rule that decides most of the credibility of a security report: **a finding that did not reproduce may not be Critical.** Cap it at Medium and say so in the same sentence, so the reader never has to cross-reference an appendix to learn that the headline finding is theoretical.

The contract enforces the habit by requiring `severity_inputs` next to every severity. One overclaimed Critical costs more trust than ten honest Lows.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/appsec/appsec-triage-report/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: appsec-triage-report
description: >-
  Calibrate severity and write the security report a team will actually act on.
  Use when asked to write up findings, assign or justify severity, produce a
  pentest or code-review report, summarise a scan for engineers or leadership,
  or decide what to escalate.
allowed-tools: Read, Write
---

# AppSec pipeline · Phase 5 — Reporting

Covers **stage 15**. Every earlier stage is measured here: a defect that was
found, deduplicated, verified and reproduced still counts for nothing if the
report gets it fixed slowly or not at all.

## When to use this

At the end of a review, or whenever findings must be handed to someone who did
not do the analysis.

## Inputs

`findings` (Phase 3), `validations` (Phase 4), and `plan.deferred` (Phase 2).
The last one is not optional — it is the scope statement.

## Procedure

**Calibrate severity, and show the calibration.** Severity is a function of
impact and reachability, and both are already computed upstream. State them:

| Severity | Requires |
|---|---|
| Critical | reproduced, unauthenticated path, sink is subprocess/deserialisation/credential |
| High | reproduced, or confirmed with an authenticated path to a dangerous sink |
| Medium | confirmed, feasible, but mitigated in depth or requires elevated access |
| Low | confirmed, not feasible in this deployment |
| Informational | needs_human, or hardening with no demonstrated path |

**A finding that did not reproduce may not be Critical.** If Phase 4 set
`reproduced: false`, cap it at Medium and say why in the same sentence — the
reader must not have to cross-reference an appendix to learn that the headline
finding is theoretical.

**Separate demonstrated from asserted.** Two sections, always. The credibility
of the whole report comes from the reader being able to tell them apart without
effort, and one overclaimed Critical costs more trust than ten honest Lows.

**Write for the person who will fix it.** Each finding needs: where it is, what
an attacker does with it, the observable that proves it, the fix, and what the
fix costs. Lead with the fix — the reader is deciding what to do, not learning
the CWE taxonomy.

**State the scope honestly.** What was analysed, what was deferred and why,
what the tooling cannot see. A report that hides its gaps invites the reader to
treat silence as coverage.

**Report accuracy, never conformance.** "100% schema-valid" is a statement
about the serialiser and is true of an empty result. If a false-positive rate
is known — Phase 4 measures one every run — report that instead.

## Output contract

```json
{
  "report": {
    "summary": {"critical": 0, "high": 0, "medium": 0, "low": 0, "informational": 0},
    "demonstrated": [
      {"finding_id": "str", "severity": "critical|high|medium|low|informational",
       "severity_inputs": {"reproduced": true, "auth": "none|user|admin", "sink": "str"},
       "title": "str", "impact": "str", "observable": "str",
       "fix": "str", "fix_cost": "low|medium|high"}
    ],
    "asserted": [{"finding_id": "str", "severity": "str", "why_not_demonstrated": "str"}],
    "scope": {"analysed": ["str"], "deferred": ["str"], "blind_spots": ["str"]},
    "quality": {"validated": 0, "failed_to_reproduce": 0, "false_positive_rate": 0.0}
  }
}
```

## Failure modes

- **Severity without `severity_inputs`.** Unarguable, therefore unactionable.
- **Burying non-reproduction.** It belongs in the finding, not the appendix.
- **Sorting by severity alone.** Ties must break deterministically
  (`-severity_rank, cwe, file, line`) or two runs disagree and the diff between
  reports becomes unreadable.
- **Counting conformance as quality.** See above; it is the most common way an
  automated pipeline flatters itself.

## Handoff

This is the end of the pipeline. Feed `quality.false_positive_rate` back into
the next run's Phase 2 scoring — a pipeline that never learns from its own
misses repeats them at machine speed.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## 4 · Where it breaks — the uncalibrated headline

Now report the same findings without the cap.

## What you just proved

Calibration moves several findings off their rule severity: the confirmed reachable CWE-89 that chains into account takeover becomes critical, while the unreachable and unvalidated ones fall. The top-3 by rule severity and by calibration disagree. The stage table shows review with the worst precision and highest minutes per finding, and design carrying the highest escape cost despite only two findings.

## Your turn

Recalculate severity for your current open findings using confirmation and reachability alone — you do not need chaining to see the effect. The queue reorders, and the items that fall are usually the ones people have been arguing about.

---

**Next → [B2.11 · Context engineering for the pipeline](https://spbreed.github.io/cyber-commons/lessons/B2.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*